<a href="https://colab.research.google.com/github/mustapqilhasbi/Portfolio/blob/main/AI_Research_Assistant_Multi_Agent_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langgraph langchain langchain-anthropic langchain-community anthropic fastapi uvicorn streamlit arxiv python-dotenv -q

In [ ]:
# Cell 2 - Setup dengan Groq (gratis!)
!pip install groq -q

from groq import Groq
import arxiv
import os

from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

client = Groq(api_key=GROQ_API_KEY)

# Test koneksi
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Halo, tes koneksi!"}],
    max_tokens=50
)
print("Setup Groq berhasil!")
print(response.choices[0].message.content)

Setup Groq berhasil!
Halo, koneksi stabil. Saya siap membantu Anda. Ada yang bisa saya bantu?


In [ ]:
# Cell 3 - Search Agent
def search_papers(query, max_results=5):
    client_arxiv = arxiv.Client()
    search = arxiv.Search(query=query, max_results=max_results)
    results = []
    for paper in client_arxiv.results(search):
        results.append({
            "title": paper.title,
            "summary": paper.summary[:500],
            "url": paper.entry_id,
            "authors": [a.name for a in paper.authors]
        })
    print(f"Ditemukan {len(results)} paper!")
    return results

# Test search
papers = search_papers("large language models 2024")
for p in papers:
    print(f"\n- {p['title']}")

Ditemukan 5 paper!

- Double Multi-Head Attention Multimodal System for Odyssey 2024 Speech Emotion Recognition Challenge

- Is Self-knowledge and Action Consistent or Not: Investigating Large Language Model's Personality

- Large Language Models Lack Understanding of Character Composition of Words

- LLMs4OL 2024 Overview: The 1st Large Language Models for Ontology Learning Challenge

- The First Place Solution of WSDM Cup 2024: Leveraging Large Language Models for Conversational Multi-Doc QA


In [ ]:
# Cell 4 - Summarizer Agent (Groq version)
def summarize_papers(papers):
    summaries = []
    for i, paper in enumerate(papers):
        print(f"Merangkum paper {i+1}/{len(papers)}...")
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{
                "role": "user",
                "content": f"Ringkas temuan utama paper ini dalam 3 poin singkat:\n\nJudul: {paper['title']}\n\nAbstrak: {paper['summary']}"
            }],
            max_tokens=500
        )
        summaries.append(response.choices[0].message.content)
    print("\nSemua paper berhasil dirangkum!")
    return summaries

# Test summarizer
summaries = summarize_papers(papers[:2])
for i, s in enumerate(summaries):
    print(f"\n=== Paper {i+1} ===")
    print(s)

Merangkum paper 1/2...
Merangkum paper 2/2...

Semua paper berhasil dirangkum!

=== Paper 1 ===
Berikut adalah 3 poin singkat tentang temuan utama paper ini:

1. Paper ini membahas tentang sistem Double Multi-Head Attention Multimodal untuk pengenalan emosi dalam ucapan (Speech Emotion Recognition, SER) yang dikembangkan untuk Odyssey 2024 Speech Emotion Recognition Challenge.
2. Sistem ini menggunakan model pra-latih dengan teknik self-supervised untuk meningkatkan kemampuan pengenalan emosi dalam ucapan.
3. Penelitian ini bertujuan untuk mempromosikan penelitian dengan pendekatan baru dalam SER dan meningkatkan akurasi pengenalan emosi dalam ucapan.

=== Paper 2 ===
Berikut adalah 3 poin singkat tentang temuan utama paper ini:

1. Penelitian ini menyelidiki kesesuaian antara sifat kepribadian yang diklaim oleh Model Bahasa Besar (LLM) dengan perilaku nyata mereka.
2. Penelitian ini menggunakan kuesioner kepribadian konvensional untuk menangkap sifat kepribadian manusia seperti pada L

In [ ]:
# Cell 5 - Critic + Writer + Full Pipeline
def critique_papers(summaries):
    print("Menganalisis dan membandingkan paper...")
    combined = "\n\n".join([f"Paper {i+1}:\n{s}" for i, s in enumerate(summaries)])
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"Bandingkan metodologi paper-paper berikut. Identifikasi kelebihan dan kekurangan tiap pendekatan:\n\n{combined}"
        }],
        max_tokens=600
    )
    return response.choices[0].message.content

def write_report(query, summaries, critique):
    print("Menulis laporan akhir...")
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": f"""Tulis laporan riset terstruktur tentang: {query}

Ringkasan paper:
{chr(10).join(summaries)}

Analisis komparatif:
{critique}

Format: Pendahuluan, Temuan Utama, Perbandingan Metodologi, Kesimpulan."""
        }],
        max_tokens=1500
    )
    return response.choices[0].message.content

# ========= JALANKAN FULL PIPELINE =========
print("="*50)
print("AI RESEARCH ASSISTANT - FULL PIPELINE")
print("="*50)

query = "computer vision object detection 2024"

print("\n[1/4] Mencari paper...")
papers = search_papers(query, max_results=3)

print("\n[2/4] Merangkum paper...")
summaries = summarize_papers(papers)

print("\n[3/4] Menganalisis metodologi...")
critique = critique_papers(summaries)

print("\n[4/4] Menulis laporan...")
report = write_report(query, summaries, critique)

print("\n" + "="*50)
print("LAPORAN RISET SELESAI!")
print("="*50)
print(report)

AI RESEARCH ASSISTANT - FULL PIPELINE

[1/4] Mencari paper...
Ditemukan 3 paper!

[2/4] Merangkum paper...
Merangkum paper 1/3...
Merangkum paper 2/3...
Merangkum paper 3/3...

Semua paper berhasil dirangkum!

[3/4] Menganalisis metodologi...
Menganalisis dan membandingkan paper...

[4/4] Menulis laporan...
Menulis laporan akhir...

LAPORAN RISET SELESAI!
**Laporan Riset: Computer Vision Object Detection 2024**

**Pendahuluan**

Teknologi computer vision object detection telah berkembang pesat dalam beberapa tahun terakhir, terutama dengan penggunaan model-machine learning yang canggih. Penelitian terbaru dalam bidang ini berfokus pada pengembangan sistem visi komputer universal yang dapat menangani berbagai tugas visi secara bersamaan. Laporan ini akan membahas temuan utama dari tiga paper penelitian yang terkait dengan computer vision object detection, yaitu "Universal Object Detection dengan Large Vision Model", "Evaluasi Vision-Language Model (VLM) pada Tugas Deteksi dan Segementas